# Level3 JAX PPO Full-DR Training Notebook

This notebook mirrors the Level3 JAX PPO notebook, but routes rollout, training, checkpointing, and eval through `lsy_drone_racing.control.jax_ppo_fast_full_dr`.

The key difference is that `level3_dr.toml` train-only DR is now represented in JAX scan state: thrust scale and battery sag, action latency, command response lag, observation latency, and observation noise.


In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.55")

ROOT = Path.cwd().resolve()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("repo:", ROOT)
print("python:", sys.executable)
print("XLA preallocate:", os.environ.get("XLA_PYTHON_CLIENT_PREALLOCATE"))
print("XLA memory fraction:", os.environ.get("XLA_PYTHON_CLIENT_MEM_FRACTION"))


Use the GPU pixi kernel for real training:

```text
.pixi/envs/gpu/bin/python
```

Install it if needed:

```bash
pixi run -e gpu python -m ipykernel install --user --name lsy-drone-racing-gpu --display-name "LSY Drone Racing (gpu)"
```


In [ ]:
%reload_ext autoreload
%autoreload 2

import json

import jax
import optax

from lsy_drone_racing.control import jax_ppo, jax_ppo_fast_full_dr
from lsy_drone_racing.control.jax_ppo import JaxPPOArgs, load_jax_checkpoint
from lsy_drone_racing.control.jax_ppo_fast_full_dr import (
    benchmark_fast_rollout_full_dr,
    evaluate_ppo_fast_full_dr,
    level2_validation_args,
    load_full_dr_config,
    train_ppo_fast_full_dr,
)

print("jax:", jax.__version__)
print("devices:", jax.devices())
print("optax:", optax.__version__)
print("checkpoint format:", jax_ppo.CHECKPOINT_FORMAT)
print("full DR module:", jax_ppo_fast_full_dr.__name__)


## Settings


In [ ]:
TRAIN_ENV_TOML = "level3_dr.toml"
EVAL_ENV_TOML = TRAIN_ENV_TOML
RUN_NAME = "jax_fast_full_dr_level3_ppo"
CONFIG = TRAIN_ENV_TOML

CONFIG_PATH = ROOT / "config" / TRAIN_ENV_TOML
if not CONFIG_PATH.exists():
    raise FileNotFoundError(CONFIG_PATH)
EVAL_CONFIG_PATH = ROOT / "config" / EVAL_ENV_TOML
if not EVAL_CONFIG_PATH.exists():
    raise FileNotFoundError(EVAL_CONFIG_PATH)

CHECKPOINT_DIR = ROOT / f"lsy_drone_racing/control/checkpoints/{RUN_NAME}"
MODEL_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_final.pkl"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

JAX_DEVICE = "gpu"
DEBUG_JAX_DEVICE = "cpu"

WANDB_ENABLED = True
WANDB_PROJECT_NAME = "ADR-PPO-Racing"
WANDB_ENTITY = None
WANDB_MODE = "online"

NUM_ENVS_DEBUG = 4
NUM_ENVS_TRAIN = 1024
NUM_STEPS = 32
TOTAL_TIMESTEPS_TRAIN = 300_000_000
CHECKPOINT_INTERVAL_STEPS = 10_000_000
LEARNING_RATE = 3e-4
GAMMA = 0.99
GAE_LAMBDA = 0.95
UPDATE_EPOCHS = 5
NUM_MINIBATCHES = 8
ENT_COEF = 0.02
TARGET_KL = 0.03
HIDDEN_DIM = 256

REWARD_KWARGS = dict(
    progress_coef=0.0,
    gate_stage_coef=15.0,
    gate_axis_coef=15.0,
    near_gate_coef=0.0,
    gate_bonus=120.0,
    gate_back_bonus=20.0,
    finish_bonus=160.0,
    missed_gate_penalty=0.0,
    wrong_side_penalty=6.0,
    crash_penalty=50.0,
    obstacle_coef=5.0,
    obstacle_margin=0.3,
    obstacle_clearance_coef=0.0,
    timeout_penalty=80.0,
    time_penalty=0.03,
    act_coef=0.03,
    d_act_th_coef=0.10,
    d_act_xy_coef=0.10,
    cmd_tilt_coef=1.0,
    rpy_coef=1.0,
    tilt_limit_deg=40.0,
    tilt_excess_coef=15.0,
)

def build_level3_args(**overrides):
    base = dict(
        level="level3",
        config=CONFIG,
        jax_device=JAX_DEVICE,
        wandb_project_name=WANDB_PROJECT_NAME,
        wandb_entity=WANDB_ENTITY,
        wandb_mode=WANDB_MODE,
        num_steps=NUM_STEPS,
        gamma=GAMMA,
        gae_lambda=GAE_LAMBDA,
        update_epochs=UPDATE_EPOCHS,
        num_minibatches=NUM_MINIBATCHES,
        learning_rate=LEARNING_RATE,
        ent_coef=ENT_COEF,
        target_kl=TARGET_KL,
        hidden_dim=HIDDEN_DIM,
        n_obs=2,
        **REWARD_KWARGS,
    )
    base.update(overrides)
    return JaxPPOArgs.create(**base)

MODEL_PATH


## 1. Inspect Full-DR Config


In [ ]:
inspect_args = build_level3_args(jax_device="cpu", num_envs=NUM_ENVS_DEBUG, total_timesteps=64)
full_dr_config = load_full_dr_config(inspect_args)
print(json.dumps(full_dr_config._asdict(), indent=2))


## 2. Full-DR Smoke Train

This verifies full-DR rollout, PPO update, checkpoint save/load, and eval on a tiny CPU run.


In [ ]:
smoke_args = build_level3_args(
    num_envs=2,
    num_steps=8,
    num_minibatches=2,
    update_epochs=1,
    total_timesteps=64,
    jax_device="cpu",
    hidden_dim=32,
)

smoke_path = Path("/tmp/jax_ppo_full_dr_level3_smoke.pkl")
train_ppo_fast_full_dr(smoke_args, smoke_path, wandb_enabled=False)
load_jax_checkpoint(smoke_path)
evaluate_ppo_fast_full_dr(smoke_args, n_eval=2, model_path=smoke_path, seed_start=1)


## 3. Level2 Fast2 Full-DR Architecture Check

The full validation target remains the Level2 Fast2 preset. The accepted threshold is 79% success over 100 eval seeds.


In [ ]:
level2_smoke_args = level2_validation_args(
    jax_device="cpu",
    num_envs=8,
    num_steps=4,
    num_minibatches=2,
    update_epochs=1,
    total_timesteps=64,
    hidden_dim=32,
)
level2_smoke_path = Path("/tmp/jax_ppo_full_dr_level2_smoke.pkl")
train_ppo_fast_full_dr(level2_smoke_args, level2_smoke_path, wandb_enabled=False)
evaluate_ppo_fast_full_dr(level2_smoke_args, n_eval=2, model_path=level2_smoke_path, seed_start=1)


In [ ]:
print(
    "pixi run -e gpu python scripts/validate_level2_jax_ppo_fast_full_dr.py "
    "--eval-episodes 100 --eval-seed-start 1 --success-threshold 0.79"
)


## 4. Benchmark Full-DR Level3 Rollout


In [ ]:
bench_args = build_level3_args(
    jax_device=JAX_DEVICE,
    num_envs=1024,
    num_steps=32,
    total_timesteps=1024 * 32,
    hidden_dim=64,
)
benchmark_fast_rollout_full_dr(bench_args, repeat=2, warmup=1)


## 5. Train Level3 Full-DR JAX PPO


In [ ]:
train_args = build_level3_args(
    num_envs=NUM_ENVS_TRAIN,
    num_steps=NUM_STEPS,
    total_timesteps=TOTAL_TIMESTEPS_TRAIN,
    wandb_run_name=RUN_NAME,
    wandb_run_id=RUN_NAME,
)

train_ppo_fast_full_dr(
    train_args,
    MODEL_PATH,
    wandb_enabled=WANDB_ENABLED,
    checkpoint_dir=CHECKPOINT_DIR,
    checkpoint_interval=CHECKPOINT_INTERVAL_STEPS,
)
print("saved:", MODEL_PATH)


## 6. Evaluate Level3 Full-DR Checkpoint


In [ ]:
eval_args = build_level3_args(
    config=EVAL_ENV_TOML,
    num_envs=1,
    jax_device="cpu",
)
summary = evaluate_ppo_fast_full_dr(
    eval_args,
    n_eval=5,
    model_path=MODEL_PATH,
    seed_start=eval_args.seed + 1,
)
print(json.dumps({k: v for k, v in summary.items() if k != "episodes_detail"}, indent=2))


## 7. Equivalent Terminal Command


In [ ]:
print(
    "pixi run -e gpu python scripts/train_jax_ppo_fast_full_dr_level3.py "
    "--config level3_dr.toml --wandb-enabled=True --checkpoint-interval=10000000 "
    "--model-name checkpoints/jax_fast_full_dr_level3_ppo/jax_fast_full_dr_level3_ppo_final.pkl"
)
